<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/Logistica_interna_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#==========================================
#HERRAMIENTA: Plan de logistica
#AUTOR: LFRG
#VERSION: 4 (Carga de datos optimizada, con CLS y reporte de impacto)
#==========================================

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.3 MB/s eta 0:00:00


In [ ]:


import pandas as pd

def generar_plan_logistica(ruta_entrada, ruta_salida, umbral_horas=10):
    """
    Genera un plan de logística consolidado con carga optimizada
    e incluye un reporte automático de ROI/Backtesting.
    """

    print(f"--- Iniciando Optimización (Umbral: {umbral_horas}h) ---")

    # --- 1. Carga de Datos ---
    cols_to_use = ['Maquina', 'Folio', 'Numero de parte', 'Cls', 'Cant. Piezas', 'Procesos', 'Fin Corte']
    try:
        df = pd.read_excel(ruta_entrada, usecols=cols_to_use)
    except Exception as e:
        print(f"Error en la carga: {e}")
        return

    # --- 2. Limpieza ---
    df['Fin Corte'] = pd.to_datetime(df['Fin Corte'], errors='coerce')
    df['Siguiente_Proceso'] = df['Procesos'].astype(str).str.split('-').str.get(2)
    df_clean = df.dropna(subset=['Fin Corte', 'Numero de parte', 'Siguiente_Proceso']).copy()

    if df_clean.empty:
        print("No hay datos válidos.")
        return

    # --- 3. Consolidación ---
    umbral = pd.Timedelta(hours=umbral_horas)
    lotes_finales = []
    grupos = df_clean.groupby(['Numero de parte', 'Siguiente_Proceso'])

    for _, data in grupos:
        df_grupo = data.sort_values('Fin Corte')
        lote_temp = []
        ultima_hora = None

        for _, fila in df_grupo.iterrows():
            if not lote_temp or (fila['Fin Corte'] - ultima_hora) <= umbral:
                lote_temp.append(fila)
                ultima_hora = fila['Fin Corte']
            else:
                lotes_finales.append(lote_temp)
                lote_temp = [fila]
                ultima_hora = fila['Fin Corte']
        if lote_temp: lotes_finales.append(lote_temp)

    # --- 4. Generación de Resultados ---
    resumen = []
    for lote in lotes_finales:
        df_l = pd.DataFrame(lote)
        resumen.append({
            'Hora_Movimiento_Estimada': df_l['Fin Corte'].max(),
            'Numero_de_Parte': df_l['Numero de parte'].iloc[0],
            'Cls': df_l['Cls'].iloc[0],
            'Proceso_Destino': df_l['Siguiente_Proceso'].iloc[0],
            'Cantidad_Total_Piezas': df_l['Cant. Piezas'].sum(),
            'Folios_Consolidados': ', '.join(df_l['Folio'].astype(str).unique()),
            'Maquinas_Origen': ', '.join(df_l['Maquina'].astype(str).unique()),
            '#_Folios_Consolidados': len(df_l)
        })

    df_plan = pd.DataFrame(resumen).sort_values('Hora_Movimiento_Estimada')

    # --- 5. BACKTESTING
    movimientos_originales = len(df_clean)
    movimientos_optimizados = len(df_plan)
    viajes_ahorrados = movimientos_originales - movimientos_optimizados
    porcentaje_ahorro = (viajes_ahorrados / movimientos_originales) * 100
    ratio_productividad = movimientos_originales / movimientos_optimizados

    print("\n" + "="*40)
    print(" REPORTE DE IMPACTO LOGÍSTICO")
    print("="*40)
    print(f"Viajes que se harían sin script: {movimientos_originales}")
    print(f"Viajes programados con script:  {movimientos_optimizados}")
    print(f"Viajes ELIMINADOS (Ahorro):     {viajes_ahorrados}")
    print(f"Eficiencia ganada (ROI):        {porcentaje_ahorro:.1f}%")
    print(f"Factor de productividad:        {ratio_productividad:.2f}x")
    print("="*40 + "\n")

    # --- 6. Exportación
    try:
        writer = pd.ExcelWriter(ruta_salida, engine='xlsxwriter', datetime_format='mmm d yyyy hh:mm AM/PM')
        df_plan.to_excel(writer, sheet_name='Plan_Logistica', index=False)
        worksheet = writer.sheets['Plan_Logistica']

        # Formato Condicional
        rows = len(df_plan)
        worksheet.conditional_format(1, 4, rows, 4, {'type': '3_color_scale', 'min_color': "#63BE7B", 'max_color': "#F8696B"})
        worksheet.conditional_format(1, 7, rows, 7, {'type': 'data_bar', 'bar_color': '#638EC6'})

        # Ajuste de columnas
        for i, col in enumerate(df_plan.columns):
            worksheet.set_column(i, i, max(df_plan[col].astype(str).map(len).max(), len(col)) + 2)

        writer.close()
        print(f"Archivo guardado: {ruta_salida}")
    except Exception as e:
        print(f"Error exportando: {e}")

# Ejecución
ARCHIVO_INPUT = 'Plandecorte_P2.xlsx'
ARCHIVO_OUTPUT = 'Plan_Logistica_con_ROI_31_03_P2.xlsx'
generar_plan_logistica(ARCHIVO_INPUT, ARCHIVO_OUTPUT, umbral_horas=10)

--- Iniciando Optimización (Umbral: 10h) ---

📊 REPORTE DE IMPACTO LOGÍSTICO
Viajes que se harían sin script: 328
Viajes programados con script:  216
Viajes ELIMINADOS (Ahorro):     112
Eficiencia ganada (ROI):        34.1%
Factor de productividad:        1.52x

Archivo guardado: Plan_Logistica_con_ROI_31_03_P2.xlsx
